In [ ]:
"""
Static Security Benchmark
This script measures how well Bandit classifies vulnerable and patched code pairs from CSV input.
"""

import json
import os
import sys
import re
import time
import shutil
import subprocess
import tempfile
import pandas as pd
from datetime import datetime
from pathlib import Path
from llmScan import *
from sklearn.metrics import classification_report

In [ ]:
# Define benchmark test for one full code snippet using Bandit

def _bandit_valid_issues(issues):
    return [
        issue for issue in issues
        if str(issue.get("issue_confidence", "")).upper() == "HIGH"
        and str(issue.get("issue_severity", "")).upper() in ("MEDIUM", "HIGH")
    ]


_BANDIT_CHECKED = False
_BANDIT_COMMAND = None
_BANDIT_MISSING_LOGGED = False
_BANDIT_FORCE_MODULE = False


def _is_command_available(cmd):
    """Check whether the first executable in a command list is still valid."""
    if not cmd:
        return False
    exe = cmd[0]
    if os.path.isabs(exe):
        return os.path.exists(exe) and os.access(exe, os.X_OK)
    return shutil.which(exe) is not None


def _get_bandit_command():
    """Resolve Bandit command once: prefer binary, fallback to python -m bandit."""
    global _BANDIT_CHECKED, _BANDIT_COMMAND, _BANDIT_FORCE_MODULE
    if _BANDIT_CHECKED:
        # Refresh stale cached paths in long-lived notebook kernels.
        if _is_command_available(_BANDIT_COMMAND):
            return _BANDIT_COMMAND
        _BANDIT_CHECKED = False
        _BANDIT_COMMAND = None

    _BANDIT_CHECKED = True

    if not _BANDIT_FORCE_MODULE:
        bandit_bin = shutil.which("bandit")
        if bandit_bin:
            _BANDIT_COMMAND = [bandit_bin]
            return _BANDIT_COMMAND

    try:
        probe = subprocess.run(
            [sys.executable, "-m", "bandit", "--version"],
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        if probe.returncode == 0:
            _BANDIT_COMMAND = [sys.executable, "-m", "bandit"]
            return _BANDIT_COMMAND
    except Exception:
        pass

    _BANDIT_COMMAND = None
    return _BANDIT_COMMAND


def _run_bandit(command):
    """Run Bandit and recover from broken launcher scripts once."""
    global _BANDIT_CHECKED, _BANDIT_COMMAND, _BANDIT_FORCE_MODULE
    try:
        return subprocess.run(
            command,
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
    except FileNotFoundError:
        # Common case: launcher exists but shebang points to missing interpreter.
        _BANDIT_CHECKED = False
        _BANDIT_COMMAND = None
        _BANDIT_FORCE_MODULE = True
        refreshed = _get_bandit_command()
        if not refreshed:
            raise
        retry_command = refreshed + command[len(command) - 5 :]
        return subprocess.run(
            retry_command,
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )


def test_vulnerabilities_benchmark(python_code_text):
    """
    Analyze a full Python code snippet for security vulnerabilities using Bandit.

    Returns:
        (success(bool), is_vulnerable(bool|None), tool_name(str), scan_time(float))
    """
    start_time = time.time()
    tool_name = "Bandit"
    temp_path = None

    try:
        bandit_base = _get_bandit_command()
        if not bandit_base:
            global _BANDIT_MISSING_LOGGED
            if not _BANDIT_MISSING_LOGGED:
                print(
                    "Bandit is not available in this environment. "
                    "Install with: pip install bandit (or pip install -r requirements.txt)."
                )
                _BANDIT_MISSING_LOGGED = True
            scan_time = time.time() - start_time
            return False, None, tool_name, scan_time

        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, encoding="utf-8") as tmp_file:
            tmp_file.write(str(python_code_text or ""))
            temp_path = tmp_file.name

        command = bandit_base + ["-r", temp_path, "-ll", "-iii", "--format", "json"]
        proc = _run_bandit(command)

        global last_bandit_output
        last_bandit_output = (proc.stdout or "") + "\n" + (proc.stderr or "")

        data = json.loads(proc.stdout or "{}")
        all_issues = data.get("results", [])
        valid_issues = _bandit_valid_issues(all_issues)

        is_vulnerable = len(valid_issues) > 0
        scan_time = time.time() - start_time
        return True, is_vulnerable, tool_name, scan_time

    except Exception as e:
        last_bandit_output = locals().get("last_bandit_output", str(e))
        print(f"Bandit run failed: {e}\nLast output:\n{last_bandit_output[:1000]}")
        scan_time = time.time() - start_time
        return False, None, tool_name, scan_time

    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)


def label_from_prediction(success, is_vulnerable):
    if not success or is_vulnerable is None:
        return "unknown"
    return "vulnerable" if is_vulnerable else "safe"


last_bandit_output = ""

print("test_vulnerabilities_benchmark function defined successfully!")

In [ ]:
# Define CSV-driven benchmark runner
def run_benchmark_from_csv(
    csv_path,
    before_col="vulnerable_code",
    after_col="fixed_code",
    verbose=True,
):
    """
    Run Bandit benchmark from a CSV where each row contains a vulnerable/patched pair.

    Vulnerable and patched snippets are analyzed separately, each as a full snippet.
    Adds classification, correctness, and completion columns for before/after snippets per row.
    """
    df = pd.read_csv(csv_path)

    required = [before_col, after_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s): {missing}")

    output_df = df.copy()

    before_predictions = []
    after_predictions = []
    before_correct = []
    after_correct = []
    before_success = []
    after_success = []
    before_completed = []
    after_completed = []
    before_times = []
    after_times = []

    for idx, row in output_df.iterrows():
        before_code = row.get(before_col, "")
        after_code = row.get(after_col, "")

        b_completed, b_pred, _, b_time = test_vulnerabilities_benchmark(before_code)
        a_completed, a_pred, _, a_time = test_vulnerabilities_benchmark(after_code)

        b_label = label_from_prediction(b_completed, b_pred)
        a_label = label_from_prediction(a_completed, a_pred)

        expected_before = "vulnerable"
        expected_after = "safe"
        b_is_correct = b_label == expected_before
        a_is_correct = a_label == expected_after

        before_predictions.append(b_label)
        after_predictions.append(a_label)
        before_correct.append(b_is_correct)
        after_correct.append(a_is_correct)
        before_success.append(b_is_correct)
        after_success.append(a_is_correct)
        before_completed.append(b_completed)
        after_completed.append(a_completed)
        before_times.append(round(b_time, 3))
        after_times.append(round(a_time, 3))

        if verbose:
            print(
                f"Row {idx + 1}: before={b_label} (expected vulnerable), "
                f"after={a_label} (expected safe)"
            )

    output_df["expected_before"] = "vulnerable"
    output_df["expected_after"] = "safe"
    output_df["bandit_classification_before"] = before_predictions
    output_df["bandit_classification_after"] = after_predictions
    output_df["bandit_correct_before"] = before_correct
    output_df["bandit_correct_after"] = after_correct
    output_df["bandit_success_before"] = before_success
    output_df["bandit_success_after"] = after_success
    output_df["bandit_completed_before"] = before_completed
    output_df["bandit_completed_after"] = after_completed
    output_df["bandit_scan_time_before_seconds"] = before_times
    output_df["bandit_scan_time_after_seconds"] = after_times
    output_df["bandit_pair_correct"] = output_df["bandit_correct_before"] & output_df["bandit_correct_after"]

    return output_df

print("run_benchmark_from_csv function defined successfully!")

In [ ]:
# Run the CSV benchmark
print(f"Starting Bandit CSV Security Benchmark on {input_csv_path}...")

# Update this path to your benchmark input CSV
input_csv_path = "data1551redacted.csv"

bandit_cmd = _get_bandit_command()
if not bandit_cmd:
    raise RuntimeError(
        "Bandit is not installed in the current environment. "
        "Install with: pip install bandit (or pip install -r requirements.txt)."
    )

df_results = run_benchmark_from_csv(input_csv_path, verbose=False)

model_tag = "Bandit"
date_stamp = datetime.now().strftime("%Y%m%d")
output_file = f"benchmark{model_tag}{date_stamp}.csv"
df_results.to_csv(output_file, index=False)

print(f"Benchmark completed for {len(df_results)} rows.")
print(f"Results saved to: {output_file}")

In [ ]:
# Calculate and display comprehensive metrics
from sklearn.metrics import classification_report
print("=" * 60)
print("BANDIT CSV BENCHMARK RESULTS")
print("=" * 60)

total_rows = len(df_results)
print(f"Total row pairs analyzed: {total_rows}")
total_predictions = total_rows * 2 if total_rows else 0
failed_before = int((df_results["bandit_completed_before"] != True).sum()) if total_rows else 0
failed_after = int((df_results["bandit_completed_after"] != True).sum()) if total_rows else 0
tests_failed = failed_before + failed_after
print(f"Tests failed: {tests_failed}/{total_predictions}")

y_true = ["vulnerable"] * total_rows + ["patched"] * total_rows
y_pred_raw = df_results["bandit_classification_before"].tolist() + df_results["bandit_classification_after"].tolist()
y_pred = ["patched" if label == "safe" else label for label in y_pred_raw]

print("\n" + "-" * 60)
print("Classification Report (total predictions)")
print("-" * 60)
print(classification_report(y_true, y_pred, labels=["vulnerable", "patched"], target_names=["vulnerable", "patched"], zero_division=0, digits=3))

# Build confusion matrix directly from y_true/y_pred to avoid correctness-column artifacts.
pred_series = pd.Series(y_pred, name="Predicted")
pred_series = pred_series.where(pred_series.isin(["vulnerable", "patched"]), "unknown")
actual_series = pd.Series(y_true, name="Actual")

confusion_matrix = pd.crosstab(actual_series, pred_series, dropna=False)
display_cols = ["vulnerable", "patched", "unknown"]
present_cols = [c for c in display_cols if c in confusion_matrix.columns]
confusion_matrix = confusion_matrix.reindex(index=["vulnerable", "patched"], columns=present_cols, fill_value=0)
confusion_matrix.index = [f"Actual: {idx}" for idx in confusion_matrix.index]
confusion_matrix.columns = [f"Predicted: {col}" for col in confusion_matrix.columns]

print("-" * 60)
print("Confusion Matrix (total predictions)")
print("-" * 60)
print(confusion_matrix.to_string())
print("-" * 60)
print("Legend: rows=actual labels, columns=predicted labels")
print("=" * 60)

before_completed = df_results["bandit_completed_before"].mean() if total_rows else 0
after_completed = df_results["bandit_completed_after"].mean() if total_rows else 0
print(f"Completion rate (before-code): {before_completed:.2%}")
print(f"Completion rate (after-code): {after_completed:.2%}")

avg_before_time = df_results["bandit_scan_time_before_seconds"].mean() if total_rows else 0
avg_after_time = df_results["bandit_scan_time_after_seconds"].mean() if total_rows else 0
total_before_time = df_results["bandit_scan_time_before_seconds"].sum() if total_rows else 0
total_after_time = df_results["bandit_scan_time_after_seconds"].sum() if total_rows else 0
total_time = total_before_time + total_after_time

print(f"\nTiming Statistics:")
print(f"Before-code - Total: {total_before_time:.3f}s | Average: {avg_before_time:.3f}s")
print(f"After-code  - Total: {total_after_time:.3f}s | Average: {avg_after_time:.3f}s")
print(f"Total scan time: {total_time:.3f}s")

In [ ]:
# Optional compact summary export
summary = {
    "total_row_pairs": len(df_results),
    "before_accuracy": float(df_results["bandit_correct_before"].mean()) if len(df_results) else 0.0,
    "after_accuracy": float(df_results["bandit_correct_after"].mean()) if len(df_results) else 0.0,
    "pair_accuracy": float(df_results["bandit_pair_correct"].mean()) if len(df_results) else 0.0,
    "before_success_rate": float(df_results["bandit_success_before"].mean()) if len(df_results) else 0.0,
    "after_success_rate": float(df_results["bandit_success_after"].mean()) if len(df_results) else 0.0,
    "before_completed_rate": float(df_results["bandit_completed_before"].mean()) if len(df_results) else 0.0,
    "after_completed_rate": float(df_results["bandit_completed_after"].mean()) if len(df_results) else 0.0,
    "tests_failed": int((df_results["bandit_completed_before"] != True).sum() + (df_results["bandit_completed_after"] != True).sum()) if len(df_results) else 0,
    "unknown_before": int((df_results["bandit_classification_before"] == "unknown").sum()) if len(df_results) else 0,
    "unknown_after": int((df_results["bandit_classification_after"] == "unknown").sum()) if len(df_results) else 0,
}

summary_df = pd.DataFrame([summary])
summary_file = "bandit_csv_benchmark_summary.csv"
summary_df.to_csv(summary_file, index=False)
print(f"Summary metrics saved to: {summary_file}")

In [ ]:
# Debug Bandit Input/Output from CSV rows
def debug_bandit_from_csv(
    csv_path,
    sample_pairs=1,
    start_row=0,
    random_sample=False,
    random_seed=42,
    before_col="vulnerable_code",
    after_col="fixed_code",
):
    df = pd.read_csv(csv_path)
    if df.empty:
        print("CSV is empty.")
        return

    total_rows = len(df)
    sample_pairs = int(sample_pairs) if sample_pairs is not None else 1
    if sample_pairs <= 0:
        print("sample_pairs must be >= 1")
        return

    if random_sample:
        sample_n = min(sample_pairs, total_rows)
        sampled_df = df.sample(n=sample_n, random_state=random_seed)
        selected_indices = sampled_df.index.tolist()
    else:
        if start_row < 0 or start_row >= total_rows:
            print(f"start_row {start_row} is out of range (0-{total_rows-1})")
            return
        end_row = min(start_row + sample_pairs, total_rows)
        selected_indices = list(range(start_row, end_row))

    print("=" * 60)
    print(f"BANDIT CSV DEBUG | Testing {len(selected_indices)} pair(s)")
    print("=" * 60)

    for i, row_index in enumerate(selected_indices, start=1):
        row = df.iloc[row_index]
        before_code = row.get(before_col, "")
        after_code = row.get(after_col, "")

        print("-" * 60)
        print(f"Sample {i}/{len(selected_indices)} | Row {row_index + 1}")

        b_completed, b_pred, _, b_time = test_vulnerabilities_benchmark(before_code)
        a_completed, a_pred, _, a_time = test_vulnerabilities_benchmark(after_code)
        b_label = label_from_prediction(b_completed, b_pred)
        a_label = label_from_prediction(a_completed, a_pred)
        b_success = b_label == "vulnerable"
        a_success = a_label == "safe"

        print(f"Before expected: vulnerable | predicted: {b_label} | success: {b_success} | completed: {b_completed} | time: {b_time:.3f}s")
        print(f"After expected: safe       | predicted: {a_label} | success: {a_success} | completed: {a_completed} | time: {a_time:.3f}s")
        print("Last Bandit raw output (first 1000 chars):")
        print(last_bandit_output[:1000])

# Example debug usage
input_csv_path = "sample2.csv"
debug_bandit_from_csv(input_csv_path, sample_pairs=3, start_row=90)
# debug_bandit_from_csv(input_csv_path, sample_pairs=5, random_sample=True, random_seed=42)